In [1]:
import os
import base64
from pathlib import Path
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

D:\code\LangChain-lesson\My-test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# 加载环境变量
load_dotenv()
DASHSCOPE_API_KEY=os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL=os.getenv("DASHSCOPE_BASE_URL")
if not DASHSCOPE_API_KEY or DASHSCOPE_API_KEY == "your_dashscope_api_key_here":
    raise ValueError("请设置环境变量 DASHSCOPE_API_KEY")
# 初始化模型（图像处理需要支持视觉的模型）
model = init_chat_model(
    model="qwen-vl-max",
    model_provider="openai",
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL
)
try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()
IMAGES_DIR = SCRIPT_DIR / "images"

In [13]:
def encode_image_to_base64(image_path: str) -> str:
    """将本地图像编码为 base64"""
    with open(image_path, "rb") as image_file:
        return base64.standard_b64encode(image_file.read()).decode("utf-8")
def get_mime_type(image_path: str) -> str:
    """根据文件扩展名获取 MIME 类型"""
    ext = Path(image_path).suffix.lower()
    mime_types = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp"
    }
    return mime_types.get(ext, "image/jpeg")
def create_image_message(text: str, image_path: str) -> HumanMessage:
    """
    创建包含本地图像的消息

    Args:
        text: 文字提示
        image_path: 本地图片路径

    Returns:
        HumanMessage 对象
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片文件不存在: {image_path}")

    image_base64 = encode_image_to_base64(image_path)
    mime_type = get_mime_type(image_path)

    content = [
        {"type": "text", "text": text},
        {
            "type": "image_url",
            "image_url": {"url": f"data:{mime_type};base64,{image_base64}"}
        }
    ]

    return HumanMessage(content=content)
def check_image_exists(filename: str) -> str:
    """
    检查图片是否存在，返回完整路径
    如果不存在则提示用户
    """
    image_path = IMAGES_DIR / filename
    if not image_path.exists():
        print(f"\n⚠️ 图片不存在: {image_path}")
        print(f"请将图片 '{filename}' 放入 images/ 目录")
        print("或者修改代码使用你自己的图片路径\n")
        return None
    return str(image_path)

In [16]:
def example():
    """
    让模型描述图片内容
    """
    print("\n" + "=" * 60)
    print("基本图像描述")
    print("=" * 60)

    # 检查图片是否存在
    image_path = check_image_exists("sample.jpg")
    if not image_path:
        print("跳过此示例")
        return None

    message = create_image_message(
        text="请详细描述这张图片中的内容。用中文回复。",
        image_path=image_path
    )

    print(f"📷 使用图片: {image_path}")
    print("正在分析图片...")

    response = model.invoke([message])

    print("\n🤖 描述结果：")
    print(response.content)

    """
    基于图片进行多轮问答
    """
    print("\n" + "=" * 60)
    print("图像问答")
    print("=" * 60)

    image_path = check_image_exists("sample.jpg")
    if not image_path:
        print("跳过此示例")
        return None

    questions = [
        "图片中有什么主要物体？",
        "图片的整体色调是什么？",
        "这张图片给你什么感觉？"
    ]

    messages = []

    # 首先发送图片
    initial_message = create_image_message(
        text="我会问你关于这张图片的一些问题。",
        image_path=image_path
    )
    messages.append(initial_message)

    print(f"📷 已加载图片: {image_path}")

    for question in questions:
        print(f"\n❓ 问题: {question}")

        messages.append(HumanMessage(content=question))
        response = model.invoke(messages)
        messages.append(response)

        print(f"💬 回答: {response.content}")

    """
    从图像中提取文字
    """
    print("\n" + "=" * 60)
    print("OCR 文字识别")
    print("=" * 60)

    # 需要一张包含文字的图片
    image_path = check_image_exists("text_image.jpg")
    if not image_path:
        print("提示: 请准备一张包含文字的图片用于 OCR 测试")
        print("跳过此示例")
        return None

    message = create_image_message(
        text="""请仔细查看这张图片，执行以下任务：
1. 描述图片的主要内容
2. 提取图片中所有可见的文字
3. 说明这是什么类型的图片（照片、截图、文档等）

用中文回复。""",
        image_path=image_path
    )

    print(f"📷 使用图片: {image_path}")
    print("正在进行 OCR 识别...")

    response = model.invoke([message])

    print("\n📝 识别结果：")
    print(response.content)


In [6]:
def main():
    print("\n"+"="*70)
    print("图像处理")
    print("="*70)

    example()

In [18]:
if __name__ == "__main__":
    main()


图像处理

基本图像描述
📷 使用图片: D:\code\LangChain-lesson\My-test\notebook\进阶\images\sample.jpg
正在分析图片...

🤖 描述结果：
这张图片是一幅动漫风格的插画，描绘了一位年轻男性和一只猫咪的温馨场景。

画面中的男性有着黑色的短发，发型略显凌乱，带有自然的蓬松感。他的面部轮廓柔和，五官精致，眼睛大而明亮，眼神温柔地微笑着，嘴角微微上扬，流露出一种温暖和愉悦的情绪。他穿着一件深蓝色的宽松卫衣，内搭白色衣物，整体造型简洁舒适，给人一种随性又亲切的感觉。

在他的右肩上，趴着一只毛茸茸的猫咪。这只猫的毛色以灰白为主，毛发浓密且蓬松，尤其是颈部和胸部的毛发显得特别柔软。它的眼睛是淡蓝色的，目光温和，正依偎在男性的脸颊旁，似乎在亲昵地蹭着他，表现出亲密和依赖的情感。

背景是纯蓝色的，简洁明快，突出了人物和猫咪的形象，使整个画面看起来清新、干净，充满宁静与温馨的氛围。整幅画作通过细腻的线条和柔和的色彩，传达出人与宠物之间深厚的情感联系，营造出一种温暖和谐的画面感。

图像问答
📷 已加载图片: D:\code\LangChain-lesson\My-test\notebook\进阶\images\sample.jpg

❓ 问题: 图片中有什么主要物体？
💬 回答: 这张图片中的主要物体包括：

1. 一位年轻的男性角色：他有黑色的短发，穿着深蓝色的上衣，面带微笑，表情温柔。
2. 一只猫：是一只毛茸茸的灰白相间的长毛猫，正依偎在他的肩膀上，头靠近他的耳朵。

背景是纯蓝色，突出了人物和猫的形象。整体风格为动漫或插画风格。

❓ 问题: 图片的整体色调是什么？
💬 回答: 这张图片的整体色调以**蓝色**为主。背景是明亮的天蓝色，给人一种清新、宁静的感觉。人物穿着深蓝色的上衣，与背景形成和谐的色彩呼应。此外，人物的黑色头发和猫的灰白色毛发为画面增添了层次感，但整体仍以蓝色系为核心基调，营造出柔和、温暖的氛围。

❓ 问题: 这张图片给你什么感觉？
💬 回答: 这张图片给人一种**温暖、宁静且充满治愈感**的感觉。

人物温柔的微笑和放松的姿态，搭配猫咪依偎在他肩上的亲密互动，传递出一种**安心与陪伴**的氛围。蓝色背景不仅让画面显得清新干净，也增强了平静和舒缓的情绪。猫的毛茸茸质感和柔和的眼神进一步